In [1]:
from binance_api import CryptoBinance
from pathlib import Path
import pandas as pd, numpy as np
from tqdm import tqdm
pd.options.plotting.backend = 'plotly'
c = CryptoBinance()
# c.update_one_ticker(Path('crypto_1m/BTCUSDT.xz'), True)
# s = c.get_snapshot()

In [3]:
c.download_all_crypto_1m()

  0%|          | 0/496 [00:00<?, ?it/s]

BNCBUSDT
[[1789387200000, '5.40000000', '5.52000000', '5.36000000', '5.50000000', '55058.00000000', 1789387259999, '298327.07700000', 614, '25124.20000000', '136401.10900000'], [1789387260000, '5.50000000', '5.69000000', '5.44000000', '5.61000000', '36876.00000000', 1789387319999, '205943.84900000', 781, '23132.10000000', '129604.99500000'], [1789387320000, '5.61000000', '5.61000000', '5.44000000', '5.48000000', '46738.30000000', 1789387379999, '257221.64400000', 946, '10886.10000000', '60092.96500000'], [1789387380000, '5.48000000', '5.52000000', '5.40000000', '5.45000000', '29768.30000000', 1789387439999, '162075.69600000', 293, '9748.40000000', '53181.74500000'], [1789387440000, '5.45000000', '5.60000000', '5.40000000', '5.54000000', '17662.00000000', 1789387499999, '96420.93500000', 309, '12001.00000000', '65650.54100000'], [1789387500000, '5.54000000', '5.55000000', '5.45000000', '5.52000000', '13090.60000000', 1789387559999, '71862.30100000', 162, '9008.20000000', '49542.84800000

 99%|█████████▉| 490/496 [00:02<00:00, 176.88it/s]

BNCBUSDT
[[1789627200000, '4.93000000', '4.93000000', '4.93000000', '4.93000000', '586.70000000', 1789627259999, '2892.43100000', 4, '586.70000000', '2892.43100000'], [1789627260000, '4.93000000', '4.93000000', '4.93000000', '4.93000000', '493.10000000', 1789627319999, '2430.98300000', 8, '493.10000000', '2430.98300000'], [1789627320000, '4.93000000', '4.93000000', '4.93000000', '4.93000000', '0.00000000', 1789627379999, '0.00000000', 0, '0.00000000', '0.00000000'], [1789627380000, '4.92000000', '4.93000000', '4.92000000', '4.93000000', '536.00000000', 1789627439999, '2639.23900000', 15, '442.60000000', '2179.71100000'], [1789627440000, '4.93000000', '4.93000000', '4.93000000', '4.93000000', '0.00000000', 1789627499999, '0.00000000', 0, '0.00000000', '0.00000000'], [1789627500000, '4.93000000', '4.93000000', '4.93000000', '4.93000000', '0.00000000', 1789627559999, '0.00000000', 0, '0.00000000', '0.00000000'], [1789627560000, '4.93000000', '4.93000000', '4.93000000', '4.93000000', '176.

KeyboardInterrupt: 

### generate bin table from downloaded crypto

In [ ]:
df: dict[str, pd.DataFrame] = {ind: val.open for ind, val in pd.read_pickle('ffilled_crypto.pkl').items()}
df2 = {}
shapes = [[ind, i.shape[0]] for ind, i in df.items()]
shapes = sorted(shapes, key=lambda a:a[1])[::-1]
shapes = [i[0] for i in shapes]
df2 = {i: df[i] for i in shapes}
table = pd.concat(df2.values(), axis=1).reindex(df2['ETHUSDT'].index)
table.columns = df2.keys()
table.to_pickle('full_table.pkl')

In [ ]:
df = pd.read_pickle('full_table.pkl')
df[df.isna()] = -1
df.to_numpy().tofile('crypto_prices.bin')

### make ticker list

In [ ]:
cols = df.columns.to_list()
cols[-1] = '?USDT'
cols[390] = '??USDT'
cols = [i[:-4] for i in cols]
max_len = max(len(i) for i in cols)
cols_np = np.array(cols, f'S{max_len + 1}')
cols_np.tofile('crypto_tickers.bin')
max_len, df.shape

### determine start, end, and duration

In [ ]:
l = df.shape[0]
start_indexes = np.array([0, *[df[ticker].value_counts().loc[-1] for ticker in tqdm(df.columns[1:])]], np.uint32)
start_indexes.tofile('start.bin')
np.array([l for i in df.columns], np.uint32).tofile('end.bin')
np.array([l-i for i in start_indexes], np.uint32).tofile('duration.bin')

In [ ]:
pd.DataFrame({'ts': pd.read_pickle('full_table.pkl').index.tz_localize(None), 'index': np.arange(df.shape[0])}).set_index('index').to_pickle('ts_lookup.pkl')

In [ ]:
df = pd.read_pickle('full_table.pkl')

In [13]:
info = c.session.get(f"{c.base}/api/v3/exchangeInfo", timeout=30).json()
info = [
    x["symbol"]
    for x in info["symbols"]
    if x["quoteAsset"] == "USDT"
]